# Week 2 · Notebook 02 — Detrending and the Box Least Squares Search
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/02_detrend_and_bls.ipynb)

**Goal:** flatten stellar variability without destroying transits, then *find* the planet with BLS.
You will build each step by hand first, then use the package version. Understand it, then automate it.

**Reading:**
- Lightkurve: [Identifying transiting planet signals](https://lightkurve.github.io/lightkurve/tutorials/3-science-examples/exoplanets-identifying-transiting-planet-signals.html)
- Hippke et al. 2019, *Wotan* ([arXiv:1906.00966](https://arxiv.org/abs/1906.00966)) — Sections 1–2
- Kovács, Zucker & Mazeh 2002, the BLS paper ([arXiv:astro-ph/0206099](https://arxiv.org/abs/astro-ph/0206099)) — Intro + Sec. 2

In [ ]:
# --- Setup: run this cell first. Works in Google Colab and locally. ---
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR fork
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})
print("ready, repo root:", ROOT)

In [ ]:
from transitpipe import io, detrend, search, plotting, inject
lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3, 4])
lc.meta.update(rstar=1.065, rstar_err=0.009)
print(len(lc), "points")

## 1. Detrending by hand: a sliding median
For each point, take the **median** of all points within ±½ window, call it the trend, divide it out.
Why median and not mean? The median ignores a few low points — like a transit.

In [ ]:
def my_median_detrend(time, flux, window_days):
    trend = np.empty_like(flux)
    for i in range(len(time)):
        near = np.abs(time - time[i]) < window_days / 2     # boolean mask!
        trend[i] = np.median(flux[near])
    return flux / trend, trend

my_flat, my_trend = my_median_detrend(lc.time, lc.flux, 0.5)
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(lc.time, lc.flux, ".", ms=1, color="k"); ax[0].plot(lc.time, my_trend, "C1")
ax[1].plot(lc.time, my_flat, ".", ms=1, color="k")
ax[0].set_ylabel("raw"); ax[1].set_ylabel("flattened"); ax[1].set_xlabel("Time [d]")
plt.show()

The package version does the same thing faster, and handles data gaps (never smooth across
a gap between quarters!). Check that they agree:

In [ ]:
flat, trend = detrend.flatten(lc, method="median", window_days=0.5, return_trend=True)
print("max difference vs your version:", np.nanmax(np.abs(flat.flux - my_flat)))
flat = detrend.sigma_clip(flat, sigma=4)        # clip only UPWARD outliers. Why upward only?
print(f"noise after detrending: {detrend.estimate_noise_ppm(flat):.0f} ppm")

### The window-length trade-off
Too short → the smoother follows the transit and eats it. Too long → stellar wiggles survive.
Rule of thumb: **window ≈ 3× the longest transit you want to find.** Let's see it happen on fake
data, where we know the true depth:

In [ ]:
fake = inject.make_synthetic_lightcurve(noise_ppm=50, seed=3)
fake_inj, truth = inject.inject_transit(fake, period=5.0, rp_rearth=4.0, seed=1)
core = detrend.transit_mask(fake.time, 5.0, truth["t0"], truth["duration"], pad=0.6)
print(f"true depth {truth['depth']*1e6:.0f} ppm, transit lasts {truth['duration']*24:.1f} h")
for w in [0.1, 0.2, 0.4, 0.75, 1.5, 3.0]:
    f = detrend.flatten(fake_inj, window_days=w)
    print(f"window {w:4.2f} d -> measured depth {(1-f.flux[core].mean())*1e6:6.0f} ppm,"
          f" noise {detrend.estimate_noise_ppm(f):5.0f} ppm")

### ✏️ Your turn 1
Explain the table above in two sentences. Notice that even the "good" windows lose ~20% of the depth on this
spotted star: in a window where the star is brightening or fading, the in-transit points drag the median down.
Week 3 shows the fix (mask the transits, detrend again). Then try `method="savgol"` and `method="biweight"`.
Which one keeps the depth best at short windows? (Hippke et al. 2019 reached the same conclusion
on thousands of real stars.)

## 2. BLS by hand (tiny version)
BLS asks: *for each trial period, if I fold the data, is there a box-shaped dip?*
Here is a deliberately slow, honest version — then we switch to the fast one.

In [ ]:
def my_bls(time, flux, periods, duration, n_phase=100):
    best = np.zeros(len(periods))
    for i, P in enumerate(periods):
        phase = (time % P) / P
        for ph in np.linspace(0, 1, n_phase, endpoint=False):
            d = np.abs(phase - ph); d = np.minimum(d, 1 - d) * P   # days from box centre
            inside = d < duration / 2
            if inside.sum() < 3:
                continue
            depth = flux[~inside].mean() - flux[inside].mean()
            snr = depth / (flux.std() / np.sqrt(inside.sum()))
            best[i] = max(best[i], snr)
    return best

trial_P = np.linspace(0.5, 3.0, 400)
t0_hand = __import__("time").time()
power_hand = my_bls(flat.time, flat.flux, trial_P, duration=0.08)
print(f"hand BLS took {__import__('time').time()-t0_hand:.1f} s for only {len(trial_P)} periods")
plt.plot(trial_P, power_hand); plt.xlabel("Trial period [d]"); plt.ylabel("best SNR")
plt.title("Hand-made BLS"); plt.show()
print("best period:", trial_P[power_hand.argmax()])

## 3. The real thing: astropy BoxLeastSquares
Same idea, smarter algorithm, ~100,000 trial periods in seconds.

In [ ]:
res = search.bls(flat, min_period=0.5, max_period=10)
print(res.summary())
plotting.plot_periodogram(res); plt.show()

**Read the periodogram.** Tall peak = the planet. Smaller peaks at **P/2, 2P, 3P…** are
*aliases* (folding at 2P still stacks every other transit). The dashed lines mark them.

**SDE** (signal detection efficiency) = how many standard deviations the peak stands above the rest.
A common threshold is SDE ≈ 7–9. Week 4 measures what threshold *your* pipeline really needs.

In [ ]:
plotting.plot_fold(flat, res.period, res.t0, window_hours=6); plt.show()
print(f"Kepler-10 b published period: 0.8374907 d  |  yours: {res.period:.5f} d")

### ✏️ Your turn 2
1. Run BLS on **your own target**. Report P, depth, SNR, SDE. Does P match the published value?
2. What happens if you skip detrending and run BLS on the raw flux?
3. Fold on 2×P. Describe what you see and why.

### 🚀 Going further (full-timers)
- **Multi-planet search:** `search.search_multiple(flat, n_planets=3)` masks each found planet and
  searches again. Try it on TOI-270 (TESS, 3 planets) or Kepler-11 (6 planets!).
- **TLS:** install `transitleastsquares` and compare its SDE to BLS for a small planet
  (Hippke & Heller 2019 claim ~10% better sensitivity). Is that true for your star?
- **Odd/even test:** fit odd- and even-numbered transits separately. Different depths ⇒ likely an
  eclipsing binary at 2P, not a planet. This is a real vetting step.

### ✅ Week 2 Checkpoint
- [ ] Kepler-10 b recovered: periodogram + folded plot, committed
- [ ] Your own target: periodogram + fold + a table of (P, depth, SNR, SDE)
- [ ] One paragraph: how did window length affect depth? (Include your table.)

In [ ]:
# YOUR CODE HERE